# Create a Simple Reflex-Based Lunar Lander Agent

In this example, we will use Gymnasium, an environment to train agents via reinforcement learning (RL). We will not use RL here but just use the environment with a custom simple reflex-based agent.

## Install Gymnasium

The documentation for Gymnasium is available at https://gymnasium.farama.org/

Steps:
1. Create a new folder and open it with VS Code and install all needed Python Extensions in VS Code.
2. Create a new virtual environment (CTRL-Shift P Python Create Environment...)
3. I needed to install swig and the Python C++ headers on WSL2 via the terminal
    * `sudo apt install swig`
    * `sudo apt-get install python3-dev`
4. Install python libraries for gymnasium with the needed extras

In [248]:
# Google Colab or if you are missing a package: Uncomment the %pip command below, run this block, and restart the session.
%pip install -q "swig>=4,<5" "gymnasium[box2d,classic-control]>=1.0,<2"

Additional installs for screen capturing so environment visualizations can be converted to embedded videos.
For recording videos, I had to install
    
* `sudo apt-get install xvfb ffmpeg`

Additional python packages for screen capturing.

In [249]:
# Google Colab or if you are missing a package: Uncomment the %pip command below, run this block, and restart the session.
%pip install -q "pyvirtualdisplay>=3,<4" "gymnasium[other]>=1.0,<2"

## The Lunar Lander Environment

![Luna Lander image](https://gymnasium.farama.org/_images/lunar_lander.gif)

The documentation of the environment is available at: https://gymnasium.farama.org/environments/box2d/lunar_lander/

* Performance Measure: A reward of -100 or +100 points for crashing or landing safely respectively. We do not use
  intermediate rewards here.

* Environment: This environment is a classic rocket trajectory optimization problem. A ship needs to land safely. The space is **continuous** with
  x and y coordinates in the range [-2.5, 2.5]. The landing pad is at coordinate (0,0).

* Actuators:  According to Pontryagin’s
  maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off. There are four discrete actions available:

    - 0: do nothing
    - 1: fire left orientation engine
    - 2: fire main engine
    - 3: fire right orientation engine

* Sensors: Each observation is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

The different ranges/settings of the environment can be queried.

In [250]:
import gymnasium as gym
import numpy as np
np.set_printoptions(precision=2)

In [251]:
def query_environment(name):
    env = gym.make(name)
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {env.spec.max_episode_steps}")
    print(f"Nondeterministic: {env.spec.nondeterministic}")
   # print(f"Reward Range: {env.reward_range}")
    print(f"Reward Threshold: {env.spec.reward_threshold}")
    env.close()

query_environment("LunarLander-v3")

Action Space: Discrete(4)
Observation Space: Box([ -2.5   -2.5  -10.   -10.    -6.28 -10.    -0.    -0.  ], [ 2.5   2.5  10.   10.    6.28 10.    1.    1.  ], (8,), float32)
Max Episode Steps: 1000
Nondeterministic: False
Reward Threshold: 200



Gymnasium environments are implemented as classes with a `make` method to create the environment, a `reset` method, and a `step` method to execute an action.
To use it with an agent function that expects percepts and returns an action, we need write glue code that connects the environment with the agent function.

In [252]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    # run one episode
    for _ in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        if verbose:
            print (f"Obs: {observation} -> Action: {action}")

        # step: execute an action in the environment
        observation, reward, terminated, truncated, info = env.step(action)

        # render the environment
        if render:
            env.render()

        if terminated:
            if verbose:
                print(f"Final Reward: {reward}")
            break

    return reward

## Example: A Random Agent

We randomly return one of the actions. The environment accepts the integers 0-3.


In [253]:
def random_agent_function(observation):
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return np.random.choice([0, 1, 2, 3], p=[0.25, 0.25, 0.25, 0.25])

Run an episode.

In [254]:
env = gym.make("LunarLander-v3", render_mode="human")

run_episode(random_agent_function, env)

env.close()

Obs: [ 0.    1.41  0.47 -0.02 -0.01 -0.11  0.    0.  ] -> Action: 3
Obs: [ 0.01  1.41  0.48 -0.05 -0.01 -0.15  0.    0.  ] -> Action: 1
Obs: [ 0.01  1.41  0.47 -0.08 -0.02 -0.11  0.    0.  ] -> Action: 3
Obs: [ 0.02  1.41  0.49 -0.1  -0.03 -0.16  0.    0.  ] -> Action: 1
Obs: [ 0.02  1.4   0.48 -0.13 -0.03 -0.13  0.    0.  ] -> Action: 0
Obs: [ 0.03  1.4   0.48 -0.16 -0.04 -0.13  0.    0.  ] -> Action: 1
Obs: [ 0.03  1.39  0.47 -0.18 -0.04 -0.1   0.    0.  ] -> Action: 2
Obs: [ 0.04  1.39  0.46 -0.18 -0.05 -0.1   0.    0.  ] -> Action: 2
Obs: [ 0.04  1.39  0.47 -0.14 -0.05 -0.1   0.    0.  ] -> Action: 2
Obs: [ 0.05  1.38  0.46 -0.13 -0.06 -0.11  0.    0.  ] -> Action: 0
Obs: [ 0.05  1.38  0.46 -0.16 -0.07 -0.11  0.    0.  ] -> Action: 3
Obs: [ 0.06  1.38  0.47 -0.18 -0.07 -0.15  0.    0.  ] -> Action: 2
Obs: [ 0.06  1.37  0.49 -0.16 -0.08 -0.14  0.    0.  ] -> Action: 3
Obs: [ 0.07  1.37  0.5  -0.18 -0.09 -0.18  0.    0.  ] -> Action: 3
Obs: [ 0.07  1.36  0.51 -0.21 -0.1  -0.23  0.   

Gymnasium displays environments using `render()` method on the local display. Headless installations like Google Colab do not have a display, but the output can be captured using a virtual display as a video and then add the video to the notebook.

We need to start a virtual display.

Now we can use wrappers for the environment to record the display. The functions are provided in
[display_record.py].



In [255]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py",
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Artificial_Intelligence/refs/heads/master/Agents/")

In [256]:
import gymnasium_display_recorder as dr

env = dr.gym_make('LunarLander-v3', 'LL1', render_fps=30)
run_episode(random_agent_function, env, verbose=False)
env.close()

dr.show('LL1')

/usr/local/lib/python3.13/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [257]:
def run_episodes(agent_function, env, n=1000):
    """Run multiple episodes with the given agent and return the rewards for each episode."""
    return [run_episode(agent_function, env, verbose=False, render=False) for _ in range(n)]

In [258]:
env = gym.make("LunarLander-v3", render_mode=None)

rewards = run_episodes(random_agent_function, env)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

## A Simple Reflex-Based Agent

To make the code easier to read, we use enumerations for actions (integers) and observations (index in the observation vector).

In [259]:
from enum import Enum

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7


Define a simple agent that uses the main thruster to reduce the falling speed if it gets too fast.

In [260]:
def rocket_agent_function(observation):
    """A simple agent function."""

    # run the main thruster, if the lander is falling too fast
    if observation[Obs.VY.value] < -.4:
        return Act.MAIN.value

    return Act.NO_OP.value

In [261]:
env = dr.gym_make('LunarLander-v3', 'LL2', render_fps=30)
run_episode(rocket_agent_function, env, verbose = False)
env.close()

dr.show('LL2')

## Evaluating the Agent

Run the agent on 100 problems and report the average reward.

Run experiments.

In [262]:
env = gym.make("LunarLander-v3", render_mode=None)

rewards = run_episodes(rocket_agent_function, env)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 100,

This is not great performance!


## Implement A Better Reflex-Based Agent

Build a better that uses its right and left thrusters to land the craft (more) safely. Test your agent function using 100 problems.

**Goal: balance the rocket and safely lane.**

**cases:**

- Basics:

  - if L + R contact = True, turn off boosts NO_OP (0)
  - if rocket fall to fast (Obs.VY.value < -.2 to -.4) turn on main boost MAIN(2)

- Balance problems:
  - Angle = 0: perfectly balance
  - Angle > 0: tilt to the right (turn on the R boost)
  - Angle < 0: Tilt to the left (turn on the L boost)

- Sliding into another direction:
  - if X < 0: turn on L
  - if X > 0: turn on R
  - if mid: everything fine




In [263]:
def better_rocket_agent_function(observation):
    #Initiate variables for easy access
    x = observation[Obs.X.value]
    y = observation[Obs.Y.value]
    vx = observation[Obs.VX.value]
    vy = observation[Obs.VY.value]
    angle = observation[Obs.ANGLE.value]
    angular_velocity = observation[Obs.ANGULAR_VELOCITY.value]
    left_leg_contact = observation[Obs.LEFT_LEG_CONTACT.value]
    right_leg_contact = observation[Obs.RIGHT_LEG_CONTACT.value]

    # If L/R leg touched ground, stop engine
    if left_leg_contact or right_leg_contact:
        return Act.NO_OP.value

    # Cases for main engine (If it fall to fast or almost hit the ground, turn it on)
    if y < 0.5 and vy < -0.15: # Almost hit the ground, turn on main engine
        return Act.MAIN.value
    if vy < -0.4:              # Slow fall
        return Act.MAIN.value

    # Blancing the angles
    if angle > 0.05 or angular_velocity > 0.05:
        return Act.RIGHT.value
    elif angle < -0.05 or angular_velocity < -0.05:
        return Act.LEFT.value

    # Anti-drift (If it turn left or right so fast, turn on to slow it down)
    if vx > 0.3:
        return Act.LEFT.value
    elif vx < -0.3:
        return Act.RIGHT.value

    # reposition back to 0,0
    if x < -0.1 and vx < 0.1:
        return Act.RIGHT.value
    elif x > 0.1 and vx > -0.1:
        return Act.LEFT.value

    return Act.NO_OP.value

In [270]:
import os
import shutil

reward = -100
attempts = 0

print("FINDING LANED ATTEMPS")

while reward < 100:
    attempts += 1
    # Create folder to store video
    video_folder = f'LL_LANED'
    env = dr.gym_make('LunarLander-v3', video_folder, render_fps=30)
    # Run agent
    reward = run_episode(better_rocket_agent_function, env, verbose=False)
    env.close()
    if reward >= 100:
        # If laned successfully, show the video
        dr.show(video_folder)
        break
    else:
        # delete video if it's failed attemp
        if os.path.exists(video_folder):
            shutil.rmtree(video_folder, ignore_errors=True)
print(f"Took {attempts} attempts to find a 100 reward one!")

FINDING LANED ATTEMPS


/usr/local/lib/python3.13/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Took 1 attempts to find a 100 reward one!


In [265]:
env = gym.make("LunarLander-v3", render_mode=None)

rewards = run_episodes(better_rocket_agent_function, env)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[100, -100, 100, -100, -100, -100, -100, -100, -100, -100, -100, 100, 100, 100, -100, 100, -100, 100, -100, 100, 100, 100, -100, -100, 100, 100, 100, -100, 100, -100, -100, 100, -100, -100, -100, -100, 100, -100, 100, -100, 100, -100, -100, 100, 100, 100, -100, -100, 100, -100, -100, 100, -100, 100, -100, 100, -100, 100, -100, 100, 100, -100, -100, -100, -100, -100, -100, 100, -100, -100, -100, 100, -100, -100, 100, -100, -100, -100, 100, 100, -100, -100, -100, -100, 100, 100, -100, 100, 100, -100, -100, 100, 100, -100, -100, 100, 100, 100, 100, -100, -100, 100, 100, -100, 100, 100, 100, 100, 100, -100, 100, -100, -100, 100, 100, -100, -100, -100, 100, -100, 100, 100, 100, -100, -100, -100, -100, 100, -100, -100, 100, 100, -100, 100, -100, -100, 100, 100, -100, 100, -100, 100, -100, -100, 100, -100, 100, 100, -100, 100, -100, 100, 100, 100, -100, -100, -100, 100, 100, 100, -100, -100, 100, 100, -100, 100, -100, -100, 100, 100, 100, -100, 100, 100, -100, -100, 100, 100, 100, 100, 100, 1